## Imports

In [36]:
import os
import shutil
import gc
import random
import numpy as np
import pandas as pd
import dotenv

from datasets import load_dataset
from huggingface_hub import snapshot_download
from pathlib import Path
from torchvision import transforms
import torchvision
from PIL import Image

from tqdm import tqdm
from torchvision.utils import save_image

## Loading & creating env variables

In [37]:
dotenv.load_dotenv()

HF_DATASET = os.getenv("HF_DATASET_NAME")
HF_TOKEN = os.getenv("HF_TOKEN")
HF_DATASET_LINK = os.getenv("HF_DATASET_LINK")

HF_SPLIT = "train"

LOCAL_DOWNLOAD_DIR = "./originalImages"

## Downloading HuggingFace dataset

In [38]:
dfOriginal = pd.read_parquet(HF_DATASET_LINK)
df = dfOriginal[["cirenId","dvBarrierEquivalentSpeedDescription"]]
df = df.drop_duplicates(keep='first')  
nulls = df["dvBarrierEquivalentSpeedDescription"].isna()
unknowns = df["dvBarrierEquivalentSpeedDescription"] == "Unknown"
df["dvBarrierEquivalentSpeedDescription"] = (
    df["dvBarrierEquivalentSpeedDescription"]
    .str.extract(r'(\d+)') # Busca uno o más dígitos
    .astype(float)         # Convierte a float (o int si estás seguro de que no hay decimales)
)
#df = dfOriginal[["cirenId","totalDeltaVKph"]]
#df = df.drop_duplicates(keep='first')  
#nulls = df["totalDeltaVKph"].isna()
missing_cirenIds = df.loc[nulls, "cirenId"].tolist()
missing_equivalent = df.loc[unknowns, "cirenId"].tolist()
imgIgnore = [f"**/CIREN/{ciren_id}/*" for ciren_id in missing_cirenIds]
imgIgnore = imgIgnore + [f"**/CIREN/{ciren_id}/*" for ciren_id in missing_equivalent]
patternIgnore = [".git*", "README.md"] + imgIgnore
#rint (len(missing_cirenIds))

print (patternIgnore)

['.git*', 'README.md', '**/CIREN/19/*', '**/CIREN/20/*', '**/CIREN/30/*', '**/CIREN/34/*', '**/CIREN/35/*', '**/CIREN/66/*', '**/CIREN/99/*', '**/CIREN/121/*', '**/CIREN/150/*', '**/CIREN/214/*', '**/CIREN/216/*', '**/CIREN/262/*', '**/CIREN/264/*', '**/CIREN/299/*', '**/CIREN/352/*', '**/CIREN/363/*', '**/CIREN/418/*', '**/CIREN/426/*', '**/CIREN/427/*', '**/CIREN/439/*', '**/CIREN/536/*', '**/CIREN/537/*', '**/CIREN/567/*', '**/CIREN/655/*', '**/CIREN/687/*', '**/CIREN/798/*', '**/CIREN/828/*', '**/CIREN/853/*', '**/CIREN/871/*', '**/CIREN/873/*', '**/CIREN/915/*', '**/CIREN/931/*', '**/CIREN/938/*', '**/CIREN/948/*', '**/CIREN/1149/*']


In [39]:

print("Descargando dataset completo...")

snapshot_download(
        repo_id=HF_DATASET,
        repo_type="dataset",
        local_dir=LOCAL_DOWNLOAD_DIR,
        token=HF_TOKEN,
        ignore_patterns= patternIgnore
    )

print("Descarga completada")

Descargando dataset completo...


Fetching ... files: 5185it [04:48, 17.96it/s]

Descarga completada


## Creating transformed dataset

In [ ]:
TRAIN_DATA_PATH = "./"+LOCAL_DOWNLOAD_DIR+"/images/CIREN/"
OUTPUT_DATA_PATH = Path("trasformedImages/imagenes/CIREN")
TRANSFORM_IMG = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor()  # Convierte a Tensor [0, 1], necesario para save_image
])

raw_image_dataset = torchvision.datasets.ImageFolder(root=TRAIN_DATA_PATH, transform=None)
classes = raw_image_dataset.classes
N_PHOTOSTRANSFORMED = 3
print(len(raw_image_dataset))

5181


In [41]:
for img_path_str, class_idx in raw_image_dataset.samples:
    img_path = Path(img_path_str)
    class_name = classes[class_idx]
    target_folder = OUTPUT_DATA_PATH / class_name
    target_folder.mkdir(parents=True, exist_ok=True)
    
    img = Image.open(img_path).convert('RGB')
    
    # Generar múltiples variaciones aleatorias para la misma imagen
    for i in range(N_PHOTOSTRANSFORMED):
        transformed_tensor = TRANSFORM_IMG(img)
        # El nombre incluirá el número de la copia: aug_0_foto.jpg, aug_1_foto.jpg...
        output_img_path = target_folder / f"aug_{i}_{img_path.name}"
        save_image(transformed_tensor, output_img_path)
transformed_image_dataset = torchvision.datasets.ImageFolder(root=OUTPUT_DATA_PATH, transform=None)
print(len(transformed_image_dataset))



15543


## Add noise to dvBarrierEquivalentSpeedDescription

In [42]:
noise_percentage = 0.15
noise_factor = np.random.uniform(1.0 - noise_percentage, 1.0 + noise_percentage, size=len(df))

df["dvBarrierEquivalentSpeedDescription"] = df["dvBarrierEquivalentSpeedDescription"] * noise_factor
df["dvBarrierEquivalentSpeedDescription"] = df["dvBarrierEquivalentSpeedDescription"].clip(lower=0)
noise_map = df.set_index('cirenId')['dvBarrierEquivalentSpeedDescription']
dfOriginal['noisy_dvBarrierEquivalentSpeedDescription'] = dfOriginal['cirenId'].map(noise_map)
print(dfOriginal[["cirenId", "noisy_dvBarrierEquivalentSpeedDescription"]].head(30))

    cirenId  noisy_dvBarrierEquivalentSpeedDescription
0        11                                  46.315825
1        11                                  46.315825
2        11                                  46.315825
3        11                                  46.315825
4        11                                  46.315825
5        11                                  46.315825
6        11                                  46.315825
7        11                                  46.315825
8        11                                  46.315825
9        11                                  46.315825
10       11                                  46.315825
11       11                                  46.315825
12       11                                  46.315825
13       11                                  46.315825
14       11                                  46.315825
15       11                                  46.315825
16       11                                  46.315825
17       1

## Generate Parquet

In [56]:
from pathlib import Path

out_path = Path("transformedImages/parquets/CIREN/ciren_training_augmented.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)
dfOriginal.to_parquet(out_path, engine="pyarrow")